## Fine tuning phi4-mini on legal data to abstract specific points on edge devices

- This is a kaggle notebook

- Installing Libraries

In [1]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-dlchjo8f/unsloth_5f188c23debf4c52978106a63e967331
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-dlchjo8f/unsloth_5f188c23debf4c52978106a63e967331
  Resolved https://github.com/unslothai/unsloth.git to commit d8abd5b704b879dc4a69c92396c809f00c8930b3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 

In [2]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


- MODEL INITIALIZATION (16-Bit Unquantized)

In [3]:
max_seq_length = 8600   # Covers the need for 8,583 maximum token length in a sample
dtype = None   # Forces 16-bit unquantized loading (T4/P100 compatible)
load_in_4bit = False

print("Loading Phi-4-mini...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/phi-4-mini-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Loading Phi-4-mini...


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.6.8: Fast Phi3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

- LORA ADAPTER CONFIGURATION

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM during 8,600-token spikes
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


- DATASET FORMATTING

In [5]:
def formatting_prompts_func(examples):
    # This translates your JSON 'messages' into the raw text tokens the model reads
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in examples["messages"]]
    return {"text": texts}

print("Loading and mapping your dataset...")
# IMPORTANT: Update this path to wherever your dataset is mounted in Kaggle (usually /kaggle/input/...)
dataset_path = "/kaggle/input/datasets/akshat14s/legal-data-in-jsonl-file-for-llm-fine-tuning/legal_data_for_llm_fine_tuning.jsonl" 
dataset = load_dataset("json", data_files=dataset_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

Loading and mapping your dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3060 [00:00<?, ? examples/s]

- TRAINING ARGUMENTS (Optimized for 16GB GPU & Long Context)

In [6]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        # Batch size 1 prevents OOM crashes when it hits 8,500-token outliers
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, # Simulates a larger batch size of 8
        
        # We switch from max_steps to epochs. 1 epoch reads your entire dataset exactly once.
        num_train_epochs = 1, 
        
        warmup_steps = 10,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit", # 8-bit optimizer saves VRAM without losing 16-bit weight precision
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Starting the Fine-Tuning Process...")
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/3060 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting the Fine-Tuning Process...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,060 | Num Epochs = 1 | Total steps = 383
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 8,912,896 of 3,844,934,656 (0.23% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.010029
20,1.901727
30,1.947499
40,2.052100
50,2.033401
60,2.119336
70,2.203676
80,2.217162
90,2.181501
100,2.147916


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-383/tokenizer_config.json.


- INFERENCE TESTING

In [7]:
FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system", 
        "content": "You are a legal data extraction AI. Extract the requested entity from the text. If it is not present, output strictly 'Not found'."
    },
    {
        "role": "user", 
        "content": "Extract the exact value for 'Governing Law'.\n\nContract Text:\nThis employment agreement shall be construed, interpreted, and governed by the exclusive jurisdiction of the courts of Punjab, India."
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 128, use_cache = True)
response = tokenizer.batch_decode(outputs)

print("\n--- Fine-Tuned Model Live Output ---")
# Phi-4 uses slightly different stop tokens than Qwen, so we split cleanly:
print(response[0].split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip())

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Fine-Tuned Model Live Output ---
<|system|>You are a legal data extraction AI. Extract the requested entity from the text. If it is not present, output strictly 'Not found'.<|end|><|user|>Extract the exact value for 'Governing Law'.

Contract Text:
This employment agreement shall be construed, interpreted, and governed by the exclusive jurisdiction of the courts of Punjab, India.<|end|><|assistant|>Punjab, India<|end|>


In [ ]:
# 1. Save the lightweight adapter weights to Hugging Face immediately
model.push_to_hub("akshatdev14/secure-legal-abstractor-adapters", token = "YOUR_HUGGINGFACE_TOKEN")

# 2. Save the tokenizer configuration
tokenizer.push_to_hub("akshatdev14/secure-legal-abstractor-adapters", token = "YOUR_HUGGINGFACE_TOKEN")

print("Emergency Save Complete! Your training work is safe.")

README.md:   0%|          | 0.00/545 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/akshatdev14/secure-legal-abstractor-adapters


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpi5bnv4gy/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Emergency Save Complete! Your training work is safe.


- EXPORT TO HUGGINGFACE

In [11]:
model.push_to_hub_merged(
    "akshatdev14/secure-legal-abstractor-phi4-mini-16bit",
    tokenizer,
    save_method = "merged_16bit",
    # token = "YOUR_HUGGINGFACE_TOKEN"
)
print("16Bit Model successfully pushed to the hugging face hub!")

Unsloth: Restored added_tokens_decoder metadata in akshatdev14/secure-legal-abstractor-phi4-mini-16bit/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:00<01:00, 60.07s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:39<00:00, 49.95s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/akshatdev14/secure-legal-abstractor-phi4-mini-16bit`
16Bit Model successfully pushed to the hugging face hub!
